### Semiconductor Wafer Sensor Dataset – Domain Backstory & Intuition

## 1. What is this dataset about?
This dataset comes from an **industrial manufacturing environment**, most commonly a **semiconductor fabrication plant (fab)**.  
In such industries, products are created through multiple tightly controlled steps, and **sensors continuously monitor machine and process behavior** to ensure quality.

The goal of this dataset is to **analyze sensor readings to identify whether a wafer is good or faulty**.

---

## 2. What is a Wafer (in simple terms)?
A **wafer** is a **thin, round slice of silicon**.

- It acts as the **base material for making electronic chips (ICs)**
- A single wafer can contain **hundreds or thousands of chips**
- If a wafer is defective, **all chips on it may fail**, causing major financial loss

That is why wafers are monitored very carefully during manufacturing.

---

## 3. How wafer manufacturing works in industry
A wafer passes through many stages such as:
- Heating in furnaces
- Chemical treatment
- Deposition of thin layers
- Etching and polishing
- Electrical testing

Each step must stay within **very strict limits**.  
Even small deviations can damage the wafer.

---

## 4. What is a Sensor?
A **sensor** is a device that measures physical or electrical conditions, such as:
- Temperature
- Pressure
- Gas or liquid flow
- Voltage and current
- Vibration
- Chemical concentration
- Time delays or stability

Sensors **do not sit on the wafer**, but are installed on:
- Processing machines
- Chambers and furnaces
- Pipelines and valves
- Robotic handling systems
- Control panels

---

## 5. How sensor values are generated
- Sensors continuously produce signals during processing
- Signals are sampled and digitized
- Values stored in the dataset can be:
  - Raw measurements
  - Averages over time
  - Variations or deviations
  - Stability or noise indicators

---

## 6. What does each row represent?
Each **row represents one wafer** (or one complete manufacturing run for a wafer).

In machine learning terms:
- Each row = **one instance**
- Each instance = **one wafer with all its sensor readings**

---

## 7. What does each column represent?
Each column (e.g., `Sensor-1`, `Sensor-2`, …) represents:
- A specific sensor measurement
- Or a derived metric from sensor signals

Different sensors measure different aspects of the manufacturing process.

---

## 8. Why do sensor values look so different?
You will notice large variation in values:
- **Large numbers (1000–3000)** → physical measurements like temperature, power, pressure
- **Small decimal values (0.000x–0.05)** → noise, drift, deviation, or stability metrics
- **Constant or near-constant values** → fixed machine settings or reference sensors

This is normal in industrial sensor data.

---

## 9. What is the `Good/Bad` column?
This is the **final quality label** after wafer inspection and testing.

- `1` → **Good wafer** (passes quality checks)
- `-1` → **Bad wafer** (fails quality checks)

This column is the **target variable** for analysis and modeling.

---

## 10. What is actually happening behind the data?
1. Wafer is processed through machines  
2. Sensors record machine behavior during processing  
3. Wafer is tested at the end  
4. Test result (Good or Bad) is attached to sensor data  

This creates a **historical record linking process behavior to quality outcome**.

---

## 11. What problem are we trying to solve?
Yes — **we are trying to identify faulty wafers using sensor data**.

Specifically:
- Detect patterns that lead to wafer failure
- Predict bad wafers early
- Understand which sensors contribute most to defects>>feature importance with the help of random forest

---

## 12. Why Exploratory Data Analysis (EDA) is important here
EDA helps us:
- Identify abnormal sensor behavior
- Detect outliers linked to bad wafers
- Find correlations between sensors
- Remove useless or constant features
- Prepare data for predictive modeling

---

## 13. Final intuition
This dataset represents the **digital fingerprint of how a wafer was manufactured**.  
By analyzing sensor readings, we aim to **understand, predict, and prevent wafer defects**, improving yield and reducing industrial losses.


In [20]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Data Overview and Cleaning

In [21]:
df = pd.read_csv("wafer_23012020_041211.csv")
df

,Unnamed: 0,Sensor-1,Sensor-2,Sensor-3,Sensor-4,Sensor-5,Sensor-6,Sensor-7,Sensor-8,Sensor-9,...,Sensor-582,Sensor-583,Sensor-584,Sensor-585,Sensor-586,Sensor-587,Sensor-588,Sensor-589,Sensor-590,Good/Bad
0,Wafer-801,2968.33,2476.58,2216.7333,1748.0885,1.1127,100.0,97.5822,0.1242,1.5300,...,NaN,0.5004,0.0120,0.0033,2.4069,0.0545,0.0184,0.0055,33.7876,-1
1,Wafer-802,2961.04,2506.43,2170.0666,1364.5157,1.5447,100.0,96.7700,0.1230,1.3953,...,NaN,0.4994,0.0115,0.0031,2.3020,0.0545,0.0184,0.0055,33.7876,1
2,Wafer-803,3072.03,2500.68,2205.7445,1363.1048,1.0518,100.0,101.8644,0.1220,1.3896,...,NaN,0.4987,0.0118,0.0036,2.3719,0.0545,0.0184,0.0055,33.7876,-1
3,Wafer-804,3021.83,2419.83,2205.7445,1363.1048,1.0518,100.0,101.8644,0.1220,1.4108,...,NaN,0.4934,0.0123,0.0040,2.4923,0.0545,0.0184,0.0055,33.7876,-1
4,Wafer-805,3006.95,2435.34,2189.8111,1084.6502,1.1993,100.0,104.8856,0.1234,1.5094,...,NaN,0.4987,0.0145,0.0041,2.8991,0.0545,0.0184,0.0055,33.7876,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,Wafer-897,2982.87,2477.01,2315.2667,2360.1325,1.1259,100.0,90.1144,0.1160,1.4695,...,NaN,0.5003,0.0106,0.0028,2.1263,0.0153,0.0048,0.0017,31.0176,1
97,Wafer-898,3084.82,2387.42,2171.5000,1028.4440,0.7899,100.0,101.5122,0.1224,1.3603,...,NaN,0.5016,0.0130,0.0028,2.5865,0.0153,0.0048,0.0017,31.0176,-1
98,Wafer-899,2955.87,2541.89,NaN,NaN,NaN,NaN,NaN,NaN,1.4493,...,NaN,0.5023,0.0140,0.0033,2.7810,0.0153,0.0048,0.0017,31.0176,-1
99,Wafer-900,2914.86,2465.11,2210.2778,2120.5760,1.0700,100.0,95.1089,0.1230,1.5817,...,NaN,0.5026,0.0121,0.0032,2.4064,0.0153,0.0048,0.0017,31.0176,1


In [22]:
df.drop(columns=["Unnamed: 0"],inplace=True)
df

,Sensor-1,Sensor-2,Sensor-3,Sensor-4,Sensor-5,Sensor-6,Sensor-7,Sensor-8,Sensor-9,Sensor-10,...,Sensor-582,Sensor-583,Sensor-584,Sensor-585,Sensor-586,Sensor-587,Sensor-588,Sensor-589,Sensor-590,Good/Bad
0,2968.33,2476.58,2216.7333,1748.0885,1.1127,100.0,97.5822,0.1242,1.5300,-0.0279,...,NaN,0.5004,0.0120,0.0033,2.4069,0.0545,0.0184,0.0055,33.7876,-1
1,2961.04,2506.43,2170.0666,1364.5157,1.5447,100.0,96.7700,0.1230,1.3953,0.0084,...,NaN,0.4994,0.0115,0.0031,2.3020,0.0545,0.0184,0.0055,33.7876,1
2,3072.03,2500.68,2205.7445,1363.1048,1.0518,100.0,101.8644,0.1220,1.3896,0.0138,...,NaN,0.4987,0.0118,0.0036,2.3719,0.0545,0.0184,0.0055,33.7876,-1
3,3021.83,2419.83,2205.7445,1363.1048,1.0518,100.0,101.8644,0.1220,1.4108,-0.0046,...,NaN,0.4934,0.0123,0.0040,2.4923,0.0545,0.0184,0.0055,33.7876,-1
4,3006.95,2435.34,2189.8111,1084.6502,1.1993,100.0,104.8856,0.1234,1.5094,-0.0046,...,NaN,0.4987,0.0145,0.0041,2.8991,0.0545,0.0184,0.0055,33.7876,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,2982.87,2477.01,2315.2667,2360.1325,1.1259,100.0,90.1144,0.1160,1.4695,0.0071,...,NaN,0.5003,0.0106,0.0028,2.1263,0.0153,0.0048,0.0017,31.0176,1
97,3084.82,2387.42,2171.5000,1028.4440,0.7899,100.0,101.5122,0.1224,1.3603,-0.0031,...,NaN,0.5016,0.0130,0.0028,2.5865,0.0153,0.0048,0.0017,31.0176,-1
98,2955.87,2541.89,NaN,NaN,NaN,NaN,NaN,NaN,1.4493,-0.0194,...,NaN,0.5023,0.0140,0.0033,2.7810,0.0153,0.0048,0.0017,31.0176,-1
99,2914.86,2465.11,2210.2778,2120.5760,1.0700,100.0,95.1089,0.1230,1.5817,0.0118,...,NaN,0.5026,0.0121,0.0032,2.4064,0.0153,0.0048,0.0017,31.0176,1


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Columns: 591 entries, Sensor-1 to Good/Bad
dtypes: float64(494), int64(97)
memory usage: 466.5 KB


In [24]:
df.describe()

,Sensor-1,Sensor-2,Sensor-3,Sensor-4,Sensor-5,Sensor-6,Sensor-7,Sensor-8,Sensor-9,Sensor-10,...,Sensor-582,Sensor-583,Sensor-584,Sensor-585,Sensor-586,Sensor-587,Sensor-588,Sensor-589,Sensor-590,Good/Bad
count,100.000000,101.000000,98.000000,98.000000,98.000000,98.0,98.000000,98.000000,101.000000,101.000000,...,34.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000
mean,3017.993400,2485.921485,2202.189353,1490.754698,1.185049,100.0,97.363236,0.122218,1.461697,0.000286,...,74.331709,0.499322,0.013624,0.003551,2.729488,0.023375,0.014840,0.004676,77.795167,-0.881188
std,71.790535,67.809173,30.194475,462.949098,0.350921,0.0,5.589615,0.002009,0.070966,0.010566,...,41.857728,0.003482,0.004323,0.000869,0.871736,0.012008,0.007528,0.002515,54.952461,0.475124
min,2825.670000,2254.990000,2114.666700,978.783200,0.753100,100.0,83.423300,0.116000,1.317900,-0.027900,...,20.309100,0.492500,0.007600,0.002100,1.515200,0.009900,0.004800,0.001700,20.309100,-1.000000
25%,2973.300000,2445.200000,2189.966700,1111.543600,0.839150,100.0,94.974725,0.120800,1.408000,-0.006800,...,47.356000,0.497300,0.011300,0.003100,2.276400,0.013400,0.009500,0.002700,33.787600,-1.000000
50%,3005.360000,2493.030000,2200.988900,1254.730700,1.159450,100.0,99.402750,0.122250,1.454200,0.001200,...,65.127550,0.499400,0.012800,0.003400,2.554900,0.021800,0.013900,0.003800,65.036500,-1.000000
75%,3071.295000,2527.200000,2213.211100,1963.801600,1.383000,100.0,101.457800,0.123400,1.507100,0.008100,...,99.419050,0.501500,0.014700,0.003800,2.949800,0.028000,0.019200,0.005900,104.303400,-1.000000
max,3221.210000,2664.520000,2315.266700,2363.641200,2.207300,100.0,107.152200,0.126200,1.641100,0.025000,...,223.101800,0.508700,0.043700,0.008900,8.816000,0.054500,0.040100,0.015000,223.101800,1.000000


In [25]:
df.isnull().sum()

Sensor-1      1
Sensor-2      0
Sensor-3      3
Sensor-4      3
Sensor-5      3
             ..
Sensor-587    0
Sensor-588    0
Sensor-589    0
Sensor-590    0
Good/Bad      0
Length: 591, dtype: int64

In [26]:
df.fillna(0, inplace=True)
df.isnull().sum()

Sensor-1      0
Sensor-2      0
Sensor-3      0
Sensor-4      0
Sensor-5      0
             ..
Sensor-587    0
Sensor-588    0
Sensor-589    0
Sensor-590    0
Good/Bad      0
Length: 591, dtype: int64

In [27]:
df

,Sensor-1,Sensor-2,Sensor-3,Sensor-4,Sensor-5,Sensor-6,Sensor-7,Sensor-8,Sensor-9,Sensor-10,...,Sensor-582,Sensor-583,Sensor-584,Sensor-585,Sensor-586,Sensor-587,Sensor-588,Sensor-589,Sensor-590,Good/Bad
0,2968.33,2476.58,2216.7333,1748.0885,1.1127,100.0,97.5822,0.1242,1.5300,-0.0279,...,0.0,0.5004,0.0120,0.0033,2.4069,0.0545,0.0184,0.0055,33.7876,-1
1,2961.04,2506.43,2170.0666,1364.5157,1.5447,100.0,96.7700,0.1230,1.3953,0.0084,...,0.0,0.4994,0.0115,0.0031,2.3020,0.0545,0.0184,0.0055,33.7876,1
2,3072.03,2500.68,2205.7445,1363.1048,1.0518,100.0,101.8644,0.1220,1.3896,0.0138,...,0.0,0.4987,0.0118,0.0036,2.3719,0.0545,0.0184,0.0055,33.7876,-1
3,3021.83,2419.83,2205.7445,1363.1048,1.0518,100.0,101.8644,0.1220,1.4108,-0.0046,...,0.0,0.4934,0.0123,0.0040,2.4923,0.0545,0.0184,0.0055,33.7876,-1
4,3006.95,2435.34,2189.8111,1084.6502,1.1993,100.0,104.8856,0.1234,1.5094,-0.0046,...,0.0,0.4987,0.0145,0.0041,2.8991,0.0545,0.0184,0.0055,33.7876,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,2982.87,2477.01,2315.2667,2360.1325,1.1259,100.0,90.1144,0.1160,1.4695,0.0071,...,0.0,0.5003,0.0106,0.0028,2.1263,0.0153,0.0048,0.0017,31.0176,1
97,3084.82,2387.42,2171.5000,1028.4440,0.7899,100.0,101.5122,0.1224,1.3603,-0.0031,...,0.0,0.5016,0.0130,0.0028,2.5865,0.0153,0.0048,0.0017,31.0176,-1
98,2955.87,2541.89,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,1.4493,-0.0194,...,0.0,0.5023,0.0140,0.0033,2.7810,0.0153,0.0048,0.0017,31.0176,-1
99,2914.86,2465.11,2210.2778,2120.5760,1.0700,100.0,95.1089,0.1230,1.5817,0.0118,...,0.0,0.5026,0.0121,0.0032,2.4064,0.0153,0.0048,0.0017,31.0176,1


In [28]:
df.duplicated().sum()

np.int64(1)

In [29]:
df.duplicated("Sensor-1").sum()

np.int64(1)

In [30]:
df.nunique()

Sensor-1      100
Sensor-2      100
Sensor-3       50
Sensor-4       50
Sensor-5       50
             ... 
Sensor-587     35
Sensor-588     31
Sensor-589     31
Sensor-590     35
Good/Bad        2
Length: 591, dtype: int64

# EDA

In [31]:
px.histogram(df,x="Sensor-1")

# Model Training

In [32]:
df["Good/Bad"].value_counts()

Good/Bad
-1    95
 1     6
Name: count, dtype: int64

Clearly we can observe there is imbalanced in the dataset,we gonna apply SMOTE technique to handle it but firstly lets train test split as we gonna apply SMOTE only on training data

Split first → SMOTE only on training → train model → test on original data

In [33]:
from sklearn.model_selection import train_test_split

X = df.drop("Good/Bad", axis=1)
y = df["Good/Bad"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [34]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((80, 590), (21, 590), (80,), (21,))

In [35]:
# As there is only 5 minority class samples in training data, SMOTE is not the best option here.
#  however for learning purpose, we will still apply SMOTE with k_neighbors=2

In [36]:
from imblearn.over_sampling import SMOTE
smote_obj=SMOTE(k_neighbors=2,random_state=42)
X_train_smote, y_train_smote=smote_obj.fit_resample(X_train,y_train)
# it takes features and target variable as input and return oversampled features and target variable

In [37]:
X_train_smote.shape,y_train_smote.shape

((150, 590), (150,))

In [38]:
y_train_smote[y_train_smote==1].shape, y_train_smote[y_train_smote==-1].shape

((75,), (75,))

In [39]:
# As we can see 150 samples are generate out of which 75 are of -1 and 75 are of 1 class 

In [40]:
X_train_smote

,Sensor-1,Sensor-2,Sensor-3,Sensor-4,Sensor-5,Sensor-6,Sensor-7,Sensor-8,Sensor-9,Sensor-10,...,Sensor-581,Sensor-582,Sensor-583,Sensor-584,Sensor-585,Sensor-586,Sensor-587,Sensor-588,Sensor-589,Sensor-590
0,2948.090000,2480.050000,2200.988900,1054.524000,1.383000,100.0,100.180000,0.120100,1.446300,0.007300,...,0.000000,0.000000,0.501500,0.012600,0.002900,2.504500,0.033800,0.006900,0.002500,20.309100
1,3215.870000,2453.970000,2212.866700,1066.953900,0.816100,100.0,101.615600,0.120300,1.396400,0.006500,...,0.000000,0.000000,0.499900,0.012200,0.003700,2.443700,0.036400,0.017200,0.006300,47.213600
2,3006.950000,2435.340000,2189.811100,1084.650200,1.199300,100.0,104.885600,0.123400,1.509400,-0.004600,...,0.000000,0.000000,0.498700,0.014500,0.004100,2.899100,0.054500,0.018400,0.005500,33.787600
3,3091.760000,2391.560000,2315.266700,2360.132500,1.125900,100.0,90.114400,0.116000,1.610700,0.025000,...,0.001700,31.017600,0.508700,0.011600,0.003200,2.276400,0.015300,0.004800,0.001700,31.017600
4,3019.380000,2499.750000,2200.955600,1126.867800,0.786000,100.0,100.370000,0.121500,1.444500,-0.015400,...,0.008900,223.101800,0.495400,0.013900,0.003600,2.800900,0.011700,0.026200,0.008900,223.101800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,3012.502481,2453.756299,2315.266700,2360.132500,1.125900,100.0,90.114400,0.116000,1.507925,0.011971,...,0.000463,8.440889,0.502586,0.010872,0.002909,2.167147,0.015300,0.004800,0.001700,31.017600
146,3063.756825,2493.004869,2274.739297,1942.189122,1.197137,100.0,94.164801,0.117797,1.467915,0.005832,...,0.000000,0.000000,0.497940,0.011481,0.002941,2.309290,0.014032,0.012339,0.004237,98.690761
147,2915.014864,2465.139818,2210.272573,2120.083339,1.070134,100.0,95.112284,0.122999,1.581639,0.011796,...,0.000000,0.000000,0.502595,0.012101,0.003200,2.406525,0.015298,0.004811,0.001704,31.117556
148,2977.229431,2439.178561,2247.293616,2205.036161,1.089709,100.0,93.347995,0.120532,1.591924,0.016454,...,0.000599,10.935840,0.504751,0.011924,0.003200,2.360566,0.015300,0.004800,0.001700,31.017600


In [41]:
y_train_smote

0     -1
1     -1
2     -1
3      1
4     -1
      ..
145    1
146    1
147    1
148    1
149    1
Name: Good/Bad, Length: 150, dtype: int64

In [42]:
# Now we gonna train our model on this smote data,and will evaluate on original test data.

7️⃣ Important metrics for your case (Wafer fault detection)

❌ Accuracy → misleading

✅ Recall (for -1 class) → CRITICAL

✅ Precision

✅ F1-score

✅ Confusion Matrix

In [43]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,recall_score

In [44]:
models={"Decision_Tree":DecisionTreeClassifier(),
        "Random_Forest":RandomForestClassifier()}

In [45]:
evaluation ={}
def model_training(models, X_train, y_train, X_test, y_test):
    for i in range(len(models)):
        model_obj=list(models.values())[i]
        model_obj.fit(X_train, y_train)
        y_pred=model_obj.predict(X_test)
        report=recall_score(y_test, y_pred,pos_label=-1)
        evaluation[list(models.keys())[i]] = report
    return evaluation

In [46]:
model_training(models, X_train_smote, y_train_smote, X_test, y_test)

{'Decision_Tree': 1.0, 'Random_Forest': 1.0}